#Data Dictionary Copilot

**Domain:** Data Platform / Enterprise RAG

**Dataset:** AdventureWorksDW2019 schema (already available locally, MS SQL Server 2022)

**Skills:** Metadata Processing, RAG, Schema Understanding, AI Assistants, Natural-Language Database Exploration

**Tools:** Python, pyodbc, MS SQL Server 2022, ChromaDB, Sentence-Transformers, Cerebras / Groq / Ollama (free tier, via shared call_llm())

---

### Problem Statement
Analysts constantly ask 'what table has X' or 'what does this column mean' and interrupt the data engineering team. A self-serve AI copilot that understands the database schema and documentation would eliminate most of these interruptions.



In [1]:
!pip install -q pyodbc chromadb sentence-transformers openai "opentelemetry-api<=1.42.1" "opentelemetry-sdk<=1.42.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.1/350.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00


### Setup

In [2]:
import pandas as pd
# ---- AI setup: same pattern for every project, so it's one thing to remember ----
# we try cerebras first (biggest free daily limit), then groq, then a local ollama model
# if all 3 are down for some reason we just return a message instead of crashing the whole script
import os
from getpass import getpass
import requests
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# asking for api keys only if they are not already set, and using getpass so the key
# does not get printed on screen or saved inside the notebook by accident
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

cerebras_client = OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

def call_llm(prompt):
    # step 1: try cerebras, it has the biggest free tier (1M tokens/day)
    try:
        response = cerebras_client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("cerebras failed:", e)

    # step 2: cerebras down or key wrong, try groq next
    try:
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("groq failed:", e)

    # step 3: last resort, try a local ollama model if one happens to be running
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"

import pyodbc
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()

Enter your Cerebras API key: ··········
Enter your Groq API key: ··········


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 1: Connect To Sql Server And Pull The Schema

In [3]:
# 1. CONNECT TO SQL SERVER AND PULL THE SCHEMA (using AdventureWorksDW2019 locally in SSMS)
try:
    conn = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=AdventureWorksDW2019;Trusted_Connection=yes;'
    )
    columns_df = pd.read_sql("""
        SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
        FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'dbo' ORDER BY TABLE_NAME, ORDINAL_POSITION
    """, conn)
    fk_df = pd.read_sql("""
        SELECT fk.name AS fk_name, tp.name AS parent_table, tr.name AS referenced_table
        FROM sys.foreign_keys fk
        JOIN sys.tables tp ON fk.parent_object_id = tp.object_id
        JOIN sys.tables tr ON fk.referenced_object_id = tr.object_id
    """, conn)
except Exception as e:
    # if SSMS isn't reachable from this notebook, just fake a small schema so the rest of the code still runs
    print("could not connect to SQL Server (", e, "), using a small fake schema instead")
    columns_df = pd.DataFrame({
        'TABLE_NAME': ['FactInternetSales'] * 3 + ['DimCustomer'] * 3,
        'COLUMN_NAME': ['SalesOrderNumber', 'OrderDate', 'CustomerKey', 'CustomerKey', 'FirstName', 'EmailAddress'],
        'DATA_TYPE': ['nvarchar', 'datetime', 'int', 'int', 'nvarchar', 'nvarchar'],
    })
    fk_df = pd.DataFrame({'fk_name': ['FK_Sales_Customer'], 'parent_table': ['FactInternetSales'],
                           'referenced_table': ['DimCustomer']})

could not connect to SQL Server ( ('01000', "[01000] [unixODBC][Driver Manager]Can't open lib 'ODBC Driver 17 for SQL Server' : file not found (0) (SQLDriverConnect)") ), using a small fake schema instead


### Step 2: Turn The Schema Into One Text Document Per Table

In [4]:
# 2. TURN THE SCHEMA INTO ONE TEXT DOCUMENT PER TABLE
table_docs, table_names = [], []
for table in columns_df['TABLE_NAME'].unique():
    cols = columns_df[columns_df['TABLE_NAME'] == table]
    col_list = ", ".join([f"{r.COLUMN_NAME} ({r.DATA_TYPE})" for r in cols.itertuples()])
    related = fk_df[(fk_df['parent_table'] == table) | (fk_df['referenced_table'] == table)]
    related_tables = set(related['parent_table']).union(set(related['referenced_table'])) - {table}
    doc = f"Table: {table}\nColumns: {col_list}\nRelated tables: {', '.join(related_tables) if related_tables else 'none'}"
    table_docs.append(doc)
    table_names.append(table)

### Step 3: Embed And Store

In [5]:
# 3. EMBED AND STORE (free, local embeddings)
collection = chroma_client.get_or_create_collection("schema_docs")
embeddings = embedder.encode(table_docs).tolist()
collection.add(documents=table_docs, embeddings=embeddings, ids=table_names)

### Step 4: The Actual Q&A Function Analysts Would Use

In [6]:
# 4. THE ACTUAL Q&A FUNCTION ANALYSTS WOULD USE
def ask_schema_copilot(question, k=3):
    q_emb = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    context = "\n\n".join(results['documents'][0])
    prompt = f"""You are a database schema assistant for AdventureWorksDW2019. Using ONLY this schema
context, answer the question. If asked about a join, give the exact foreign key path.

Schema context:
{context}

Question: {question}
Answer:"""
    return call_llm(prompt)

# --- DEMO ---
print(ask_schema_copilot("Which table contains customer transaction/sales data?"))
print(ask_schema_copilot("How do I join FactInternetSales to DimCustomer?"))
print(ask_schema_copilot("What column stores the order date in sales facts?"))

cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
The **FactInternetSales** table contains the customer transaction/sales data.
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
To join **FactInternetSales** to **DimCustomer**, use the `CustomerKey` column that exists in both tables:

```sql
SELECT  f.*,
        d.FirstName,
        d.EmailAddress
FROM    FactInternetSales AS f
JOIN    DimCustomer      AS d
        ON f.CustomerKey = d.CustomerKey;
```

**Foreign‑key path:** `FactInternetSales.CustomerKey → DimCustomer.CustomerKey`.
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', '